<a href="https://colab.research.google.com/github/Santosh-S321/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Santosh-S321/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb, os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH_PATH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

import pandas as pd
pd.set_option("display.max_rows", None)
schema = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{MONTH_PATH}') LIMIT 1").df()
print(schema[["column_name", "column_type"]].to_string())

                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt      BIGINT
23             a

Confirmed real column names before writing queries — notably `gsc_avg_position` exists
   as a precomputed field (not something to derive manually), and AI traffic is split by
   platform (`sessions_ai`, `ai_chatgpt`, etc.) rather than one combined column.

## 1. Unit of analysis + time window

**Unit of analysis:** one row = one content page, on one report day — the grain is
`report_date × client_hash_id × content_hash_id` in `fact_content_daily_performance`.

**Table(s):** `fact_content_daily_performance`, partition `month=2026-03` (a mid-panel
month, per the warning against using the `_sample`/final month for label logic).

**Time window:** `report_date` between 2026-03-01 and 2026-03-31.

In [2]:
q1 = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM read_parquet('{MONTH_PATH}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Grain check — rows with duplicate keys (should be empty):")
print(q1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check — rows with duplicate keys (should be empty):
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []


## 2. Fields: feature / label / context / excluded

**Label/proxy (what I'd predict or rank):** a within-month decline proxy — pages where
clicks in the second half of March are lower than the first half. This is a defined,
in-window proxy (not yet a real future-window label) — formalized properly in ML-05.

**Feature bucket:** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_sessions`
(filtered on `ga4_data_available`) — all observed/trailing measurements, knowable by
the end of the March window.

**Context (never a feature):** `client_hash_id`, `content_hash_id`, `report_date` —
join/grouping keys only.

**Excluded (deliberate):** `fact_content_query_90d` is not joined in this week's slice.
Its 90-day window overlaps the snapshot's final months, so joining it now risks pulling
in information from outside March's clean window — safer to exclude until the window
alignment is checked properly (ML-05).

In [3]:
q2 = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM read_parquet('{MONTH_PATH}')
""").df()
print("Row count and date span for March 2026 slice:")
print(q2)

Row count and date span for March 2026 slice:
   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31


## 3. Verify it with queries (grain, counts, missing values, windows)

**Availability check:** filtering on `ga4_data_available IS TRUE` — how many rows
actually have usable GA4 session data in this window, versus the total slice.

In [4]:
q3 = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM read_parquet('{MONTH_PATH}')
""").df()
q3["pct_gsc_available"] = (q3["gsc_available_rows"] / q3["total_rows"] * 100).round(1)
q3["pct_ga4_available"] = (q3["ga4_available_rows"] / q3["total_rows"] * 100).round(1)
print("Availability check (IS TRUE filter, both GSC and GA4):")
print(q3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Availability check (IS TRUE filter, both GSC and GA4):
   total_rows  gsc_available_rows  ga4_available_rows  pct_gsc_available  \
0     9841378           3611061.0            413966.0               36.7   

   pct_ga4_available  
0                4.2  


**Five features (page-level, aggregated over March):**

1. `total_impressions_month` — sum of `gsc_impressions` across March. Knowable at the
   decision moment because it's a trailing sum of already-observed daily search data.
2. `total_clicks_month` — sum of `gsc_clicks` across March. Same reasoning: observed,
   trailing.
3. `avg_position_month` — mean of `gsc_avg_position` where it's non-zero (0 = no data,
   not rank zero). Knowable once search data lands for the day.
4. `days_with_impressions` — count of distinct days with `gsc_impressions > 0`. A
   trailing count of observed activity, known by month end.
5. `total_sessions_month` — sum of `ga4_sessions`, filtered to `ga4_data_available IS
   TRUE`. Observed analytics data; the availability filter avoids treating "no
   tracking yet" as zero engagement.

In [5]:
features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions_month,
        SUM(gsc_clicks) AS total_clicks_month,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_month,
        COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_with_impressions,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS total_sessions_month
    FROM read_parquet('{MONTH_PATH}')
    GROUP BY client_hash_id, content_hash_id
""").df()

print(features.shape)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(331437, 7)


,client_hash_id,content_hash_id,total_impressions_month,total_clicks_month,avg_position_month,days_with_impressions,total_sessions_month
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,4.394234,31,0.0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,7.842593,26,0.0
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,8.454069,30,4.0
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,6.320337,31,9.0
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,4.459107,31,3.0


**The trap (deliberate leak):** the label above (`is_declining`) is built from splitting
March into first-half vs second-half clicks. Watch what happens if I add
`second_half_clicks` — the exact quantity the label was built from — back in as a
feature: the score jumps toward perfect, because the model isn't learning anything,
it's just reading the answer back off itself. Removing it restores the honest number.

In [6]:
half = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS first_half_clicks,
        SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS second_half_clicks
    FROM read_parquet('{MONTH_PATH}')
    GROUP BY client_hash_id, content_hash_id
""").df()

half["is_declining"] = (half["second_half_clicks"] < half["first_half_clicks"]).astype(int)

leaky = features.merge(half, on=["client_hash_id", "content_hash_id"])

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X_leak = leaky[["total_impressions_month", "total_clicks_month", "avg_position_month",
                "days_with_impressions", "total_sessions_month", "second_half_clicks"]].fillna(0)
y = leaky["is_declining"]
m_leak = LogisticRegression(max_iter=1000).fit(X_leak, y)
auc_leak = roc_auc_score(y, m_leak.predict_proba(X_leak)[:, 1])
print(f"WITH leak (second_half_clicks included): ROC-AUC = {auc_leak:.3f}  <- looks amazing, means nothing")

X_honest = leaky[["total_impressions_month", "total_clicks_month", "avg_position_month",
                   "days_with_impressions", "total_sessions_month"]].fillna(0)
m_honest = LogisticRegression(max_iter=1000).fit(X_honest, y)
auc_honest = roc_auc_score(y, m_honest.predict_proba(X_honest)[:, 1])
print(f"WITHOUT leak: ROC-AUC = {auc_honest:.3f}  <- honest, keep this one")

WITH leak (second_half_clicks included): ROC-AUC = 1.000  <- looks amazing, means nothing
WITHOUT leak: ROC-AUC = 0.903  <- honest, keep this one


## 4. Data limits

**Named limitation:** this slice is a single mid-panel month (March 2026) from clients
with an unbalanced panel — some clients' `ga4_data_start` falls after March, so their
`total_sessions_month` is legitimately zero due to missing tracking, not zero
engagement. This slice also can't speak to seasonality or longer-term trend behavior,
since it's one 31-day window, not a multi-month history.

In [7]:
limit_check = con.sql(f"""
    SELECT COUNT(DISTINCT client_hash_id) AS clients_in_march_slice
    FROM read_parquet('{MONTH_PATH}')
""").df()
print(limit_check)

   clients_in_march_slice
0                      55


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.